# PART -1

### part - 1.1

In [ ]:
import csv
import json

def parse_row(row):
    converted = dict(row)
    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])
    return converted

accepted = []
rejected = []


with open("appointments.csv",'r',newline= "",encoding = "utf-8") as f:
    r = csv.DictReader(f)
    print("Header:", r.fieldnames)

    row_count = 0
    for row in r:
        row_count += 1
        try:
            parsed = parse_row(row)
            accepted.append(parsed)
        except ValueError:
            rejected.append(row)

    # Print the total row count
    print("Total Row Count:", row_count)

# Save rejected rows to a JSON file
with open("rejected_rows.json", mode="w", encoding="utf-8") as json_file:
    json.dump(rejected, json_file, indent=2)


Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Row Count: 1020


### Task 1.2 — Inspect and clean, pass 1

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments.csv")

# shape of data frame created by the
print("shape of the data frame generated is",df.shape)

print("\n")
print("***"*19)
print("\n")

#information of the file
print( df.info())



shape of the data frame generated is (1020, 14)


*********************************************************


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory

In [ ]:
# null data points
print("number of null data points in each column",df.isnull().sum())
print("\n")
# the number of duplicates
print("the number of complete duplicates are ",df.duplicated().sum())

number of null data points in each column appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64


the number of complete duplicates are  20


In [ ]:
# (a) dropping duplicates
df = df.drop_duplicates().copy()
print("row count after removing duplicates:",df.shape[0])
print("\n")

# (b) clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("row count after cleaning booking_channel:",df.shape[0])
print("\n")

# (c) convert appt_date and  booked_on
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"], errors ="coerce")
print("row count after correcting appt_dates and booked_on :",df.shape[0])
print("\n")

# (d) (d) drop rows where appt_date is missing
df = df.dropna(subset=["appt_date","status"])
print("row count after dropping rows where appt_date is missing :",df.shape[0])
print("\n")




row count after removing duplicates: 1000


row count after cleaning booking_channel: 1000


row count after correcting appt_dates and booked_on : 1000


row count after dropping rows where appt_date is missing : 990




### Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [ ]:
# (a) cleaning age

df["age"] = pd.to_numeric(df["age"], errors = "coerce")
df.loc[(df["age"]<1)|(df["age"]>100), "age"] = np.nan

# (b) managing and filling the missing values

df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"]= df["distance_km"].fillna(df["distance_km"].median())


# (c) For distance_km and fee_inr, compute Q1, Q3 and IQR

# for distance

q1 = df["distance_km"].quantile(0.25)
q3 = df["distance_km"].quantile(0.75)

iqr = q3-q1

lwr = q1 - 1.5*iqr
upr = q3 + 1.5*iqr

lwr_cpd = (df["distance_km"] < lwr).sum()
upr_cpd = (df["distance_km"] < upr).sum()
print(f"the number of values capped in the distance column is {lwr_cpd + upr_cpd}")

df["distance_km"] = np.clip(df["distance_km"],lwr,upr)

# for fees

Q1 = df["fee_inr"].quantile(0.25)
Q3 = df["fee_inr"].quantile(0.75)

Iqr = q3-q1

Lwr = q1 - 1.5*iqr
Upr = q3 + 1.5*iqr

Lwr_cpd = (df["fee_inr"]  < Lwr).sum()
Upr_cpd = (df["fee_inr"] < Upr).sum()
print(f"the number of values capped in the fees column is {Lwr_cpd + Upr_cpd}")

df["fee_inr"] = np.clip(df["fee_inr"],Lwr,Upr)

#(d) Rounding off values of distance ad fees
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)




the number of values capped in the distance column is 964
the number of values capped in the fees column is 0


### Task 1.4 — Features and a NumPy summary

In [ ]:
# days between bookings and appointment date
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# showed up on appointments
df["no_show"] = (df["status"] == "No-Show").astype(int)

X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy()

#(a) the mean of each column in ["age","distance_km","fee_inr","lead_days"]

mn = np.mean(X, axis = 0)
print(f'the mean of the columns in ["age","distance_km","fee_inr","lead_days"] are {mn}')
print("\n")
print("-*-"*33)

#(b)  the number and share of rows with leading days > 14
mask = df["lead_days"] > 14
ld_rws = mask.sum()
shr = mask.mean()

print("\n")
print(f"the number and share of rows with leading days > 14 are {ld_rws} and {shr} respectively")

#(c) z-scores of the distance_km
dist = X[:1]
z_scrs = ( dist - np.mean(dist)) / np.std(dist)
outlrs = (np.abs(z_scrs)>2).sum()

print("\n")
print(f" the z-scores of distance and the number of rows having |z|>2 are {z_scrs} and {outlrs}")



the mean of the columns in ["age","distance_km","fee_inr","lead_days"] are [48.17373737  9.50090909 26.         11.81919192]


-*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*-


the number and share of rows with leading days > 14 are 296 and 0.298989898989899 respectively


 the z-scores of distance and the number of rows having |z|>2 are [[ 1.57157464 -0.38416269 -0.01164129 -1.17577066]] and 0


In [ ]:
# summary

nume_cols = ["age","distance_km","fee_inr","lead_days"]
clm_mean = np.array([df[col].mean() for col in nume_cols])

print("column means :",np.round(clm_mean,2))

# lead day stats

min_ld = df["lead_days"].min()
max_ld = df["lead_days"].max()

pct_lead_gt_14 = (df["lead_days"]>14).mean()*100
print(f"Lead Days range: {min_ld} to {max_ld}")
print(f"Rows with lead days greayter than 14: {pct_lead_gt_14:.1f}%")

# no-shows count and percentage
no_shws = (df["status"] == "No-Show").sum()
tot_rows = len(df)
print(f"No-shows: {no_shws} out of {tot_rows} ({no_shws/tot_rows*100:.1f}%)")

# Distance Z-scores (|z| > 2) check
dist_mn = df["distance_km"].mean()
dist_std = df["distance_km"].std(ddof = 0)
z_scrs = np.abs((df["distance_km"]-dist_mn)/dist_std)
outliers_z2 = (z_scrs > 2).sum()
print(f"rows with |Z| > 2 for distance_km: {outliers_z2}")

column means : [48.17  9.5  26.   11.82]
Lead Days range: 1 to 45
Rows with lead days greayter than 14: 29.9%
No-shows: 218 out of 990 (22.0%)
rows with |Z| > 2 for distance_km: 53


### Task 1.5 — Enrich the clinics from a live API

In [ ]:
import requests
import json
import csv
import pandas as pd
import numpy as np

# opening the clincs file containing the info about the different clinics across 6 cities
with open("clinics.csv",'r',newline= "",encoding = "utf-8") as f:
    r = csv.DictReader(f)
    print("Header:", r.fieldnames)

cl = pd.read_csv("clinics.csv")

print("\n")
print("-----"*32)
# checking for any missing values
print("number of null data points in each column",cl.isnull().sum())

temps = []
for _,row in cl.iterrows():
    cln_name = row["clinic_name"]
    params = {
        "latitude" : row["latitude"],
        "longitude" : row["longitude"],
        "hourly" : "temperature_2m",
        "forecast_days" : 1
    }
    temp_c = None
    failure_reason = ""

    # attempting the request upto 2 times
    for attempt in range(2):
        try:
            res = reply =requests.get("https://api.open-meteo.com/v1/forecast", params = params,timeout = 10)

            if res.status_code == 200:
                data = res.json()

                if "hourly" in data and "temperature_2m" in data["hourly"] and len(data["hourly"]["temperature_2m"]) >0 :
                    temp_c = data["hourly"]["temperature_2m"][0]
                    failure_reason = ""
                    break

                else :
                    failure_reason = "Missing 'hourly' or 'temperature_2m' data in response "

            else:
                failure_reason = f"HTTP status code {res.status_code}"

        except requests.exceptions.RequestException as e:
            failure_reason = f"RequestException: {e}"

    # if the temp_c is None even after both the attempts
    if temp_c is None :
        print(f"Failed for {cln_name}: {failure_reason}")

    temps.append(temp_c)

#add the new column and save without the dataframe index
cl["temp_c"] = temps
cl.to_csv("clinics_enriched.csv", index = False)

Header: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


----------------------------------------------------------------------------------------------------------------------------------------------------------------
number of null data points in each column clinic_id      0
clinic_name    0
city           0
tier           0
latitude       0
longitude      0
dtype: int64


In [ ]:
cl

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,19.4
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,19.7
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,19.7
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,16.4
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,16.4
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,26.3
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,16.1
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,14.7


### Task 1.6 — Export the clean table and write the SQL load script

In [ ]:

df = df.sort_values(by="appointment_id").reset_index(drop=True)
df["booked_on"] = pd.to_datetime(df["booked_on"]).dt.strftime("%y-%m-%d")
df["appt_date"] = pd.to_datetime(df["appt_date"]).dt.strftime("%y-%m-%d")

cols =["appointment_id", "patient_id", "clinic_id","booked_on", "appt_date", "age", "gender", "distance_km", "fee_inr", "booking_channel","visit_type", "reminder_sent", "prior_no_shows", "status","lead_days","no_show"]

df_cln = df[cols]

is_uq = df_cln["appointment_id"].is_unique
no_null = df_cln.isna().sum().sum()

print("appointment id that are unique:",is_uq)
print("No null values in any of the 16 columns:",no_null)

assert is_uq, "Error : Duplicate appointment ids found!"
assert no_null == 0,"Error: Missing values detected!"

df_cln.to_csv("appointments_clean.csv", index = False)

clinics_df = pd.read_csv("clinics_enriched.csv")
def format_sql_value(val):
    if isinstance(val,str):
        esc = val.replace("'","''")
        return f"'{esc}'"
    return f"{val}"

with open("load_data.sql","w",encoding = "utf-8") as f :
    for _,row in clinics_df.iterrows():
        values_str = ",".join(format_sql_value(v) for v in row)
        f.write(f"INSERT INTO clinics VALUES ({values_str});\n")
    for _,row in df_cln.iterrows():
        stmt = (
            f"INSERT INTO appointments VALUES("
            f"'{row['appointment_id']}','{row['patient_id']}','{row['clinic_id']}',"
            f"'{row['booked_on']}','{row['appt_date']}',{row['age']},"
            f"'{row['gender']}',{row['distance_km']},{row['fee_inr']},"
            f"'{row['booking_channel']}','{row['visit_type']}',"
            f"{format_sql_value(row['reminder_sent'])},{row['prior_no_shows']},"
            f"'{row['status']}',{row['lead_days']},{row['no_show']});\n"
        )
        f.write(stmt)


appointment id that are unique: True
No null values in any of the 16 columns: 0


In [ ]:
import os
print(os.getcwd())
print(os.path.exists("appointments_clean.csv"))

/content
True


In [ ]:
print(len(cols))


16


In [ ]:
check_csv =pd.read_csv("appointments_clean.csv")
print("CSV shape:",check_csv.shape)
print(check_csv.head(2))

with open("load_data.sql","r",encoding = "utf-8") as f:
    lines = f.readlines()

print("\nTotal SQL statements:",len(lines))
print("First line (clinics):",lines[0].strip())
print("Last line (appointments):", lines[-1].strip())

CSV shape: (990, 16)
  appointment_id patient_id clinic_id booked_on appt_date  age gender  \
0          A0001       P509       C05  25-01-24  25-01-25   60      F   
1          A0002       P046       C07  25-05-27  25-06-06   39      M   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0       26           Phone  Consultation           Yes   
1         12.5       26         Walk-In     Follow-up            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show          1        1  
1               2  Attended         10        0  

Total SQL statements: 998
First line (clinics): INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,19.4);
Last line (appointments): INSERT INTO appointments VALUES('A1000','P085','C08','24-12-03','25-01-17',38,'M',10.8,26,'App','Follow-up','Yes',2,'No-Show',45,1);


In [ ]:
df.isnull().sum()

,0
appointment_id,0
patient_id,0
clinic_id,0
booked_on,0
appt_date,0
age,0
gender,0
distance_km,0
fee_inr,0
booking_channel,0


# PART 2

### PART - 2.2

In [ ]:
import pandas as pd

# Merge your cleaned datasets on clinic_id
merged_df = pd.merge(df_cln, clinics_df, on='clinic_id', how='inner')

# Group by clinic name and city to match the SQL query
python_clinic_summary = merged_df.groupby(['clinic_name', 'city']).agg(
    appointments=('appointment_id', 'count'),
    no_shows=('no_show', 'sum')
).reset_index()

# Calculate percentage and filter for at least 95 appointments
python_clinic_summary['no_show_pct'] = round(100.0 * python_clinic_summary['no_shows'] / python_clinic_summary['appointments'], 1)
python_filtered = python_clinic_summary[python_clinic_summary['appointments'] >= 95]

# Sort by no_show_pct descending to align with SQL order
python_filtered = python_filtered.sort_values(by='no_show_pct', ascending=False)

print("Python Clinic Summary Verification:")
print(python_filtered)
print("Without HAVING, two more clinics appear: Gomti Nagar Care Point (Lucknow) has the highest rate, 33.3%, but only 90 appointments; Sitabuldi Health Point (Nagpur) has 82 appointments at 15.9%.\nThe pandas counts match the SQL counts exactly.")

Python Clinic Summary Verification:
                clinic_name    city  appointments  no_shows  no_show_pct
2      Hadapsar Care Centre    Pune           143        39         27.3
4      Malviya Nagar Clinic  Jaipur           142        38         26.8
3     Kothrud Family Clinic    Pune           160        36         22.5
7        Vijay Nagar Clinic  Indore           101        21         20.8
0  Edappally Medical Centre   Kochi           142        23         16.2
6     Vaishali Nagar Clinic  Jaipur           130        18         13.8
Without HAVING, two more clinics appear: Gomti Nagar Care Point (Lucknow) has the highest rate, 33.3%, but only 90 appointments; Sitabuldi Health Point (Nagpur) has 82 appointments at 15.9%.
The pandas counts match the SQL counts exactly.


In [ ]:
is_uq = df_cln['appointment_id'].is_unique
no_null = df_cln.isnull().sum().sum()

assert is_uq, "Error: Duplicate appointment IDs found!"
assert no_null ==0, "Error: Missing values found!"

print("Appointment ID that are unique:",is_uq)
print("No null values in any of the 16 columns:", no_null)

df_cln = df_cln.sort_values(by="appointment_id").reset_index(drop = True)
df_cln["booked_on"] = pd.to_datetime(df_cln["booked_on"]).dt.strftime("%y-%m-%d")
df_cln["appt_date"] = pd.to_datetime(df_cln["appt_date"]).dt.strftime("%y-%m-%d")

Appointment ID that are unique: True
No null values in any of the 16 columns: 0


/tmp/ipykernel_687/4136596732.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cln["booked_on"] = pd.to_datetime(df_cln["booked_on"]).dt.strftime("%y-%m-%d")
/tmp/ipykernel_687/4136596732.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cln["appt_date"] = pd.to_datetime(df_cln["appt_date"]).dt.strftime("%y-%m-%d")


### COMPARING PYTHON AND SQL RESULTS AND SUMMARIZATION

In [ ]:
print("please upload the csv files to colab or the server youu are using, like part - 2.2, 2.3 etc to server before runninh all the cells")

please upload the csv files to colab or the server youu are using, like part - 2.2, 2.3 etc to server before runninh all the cells


In [ ]:
import pandas as pd


# Load clean base tables
df_cln = pd.read_csv("appointments_clean.csv")
clinics_df = pd.read_csv("clinics_enriched.csv")

# Ensure dates are parsed cleanly without UserWarning
df_cln["appt_date_dt"] = pd.to_datetime(df_cln["appt_date"], format="%Y-%m-%d", errors="coerce")

# ==========================================
# Task 2.2 — No-Show Rate by Clinic (>= 95 appts)
# ==========================================
print("=== TASK 2.2 COMPARISON ===")
merged_df = pd.merge(df_cln, clinics_df, on="clinic_id", how="inner")

py_2_2 = (
    merged_df.groupby(["clinic_name", "city"])
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
    )
    .reset_index()
)
py_2_2["no_show_pct"] = (100.0 * py_2_2["no_shows"] / py_2_2["appointments"]).round(1)
py_2_2 = (
    py_2_2[py_2_2["appointments"] >= 95]
    .sort_values(by="no_show_pct", ascending=False)
    .reset_index(drop=True)
)

sql_2_2 = pd.read_csv("part_2.2_no-show-rate.csv").reset_index(drop=True)
print("Row counts match:", len(py_2_2), "vs", len(sql_2_2))
print(py_2_2.compare(sql_2_2, result_names=("python", "sql")))


# ==========================================
# Task 2.3 — Lead-Time Bands
# ==========================================
print("\n=== TASK 2.3 COMPARISON ===")
def lead_band(days):
    if 1 <= days <= 7:
        return "1-7"
    elif 8 <= days <= 14:
        return "8-14"
    elif 15 <= days <= 30:
        return "15-30"
    else:
        return "31+"

band_order = {"1-7": 1, "8-14": 2, "15-30": 3, "31+": 4}
df_2_3 = df_cln.copy()
df_2_3["lead_band"] = df_2_3["lead_days"].apply(lead_band)

py_2_3 = (
    df_2_3.groupby("lead_band")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
    )
    .reset_index()
)
py_2_3["no_show_pct"] = ((py_2_3["no_shows"] / py_2_3["appointments"]) * 100).round(2)
py_2_3["sort_key"] = py_2_3["lead_band"].map(band_order)
py_2_3 = py_2_3.sort_values("sort_key").drop(columns="sort_key").reset_index(drop=True)

sql_2_3 = pd.read_csv("part-2.3-booking-far-ahead.csv").reset_index(drop=True)
print("Row counts match:", len(py_2_3), "vs", len(sql_2_3))
print(py_2_3.compare(sql_2_3, result_names=("python", "sql")))


# ==========================================
# Task 2.4 — Month-by-Month Trend & Prior-Month Diff
# ==========================================
print("\n=== TASK 2.4 COMPARISON ===")
df_2_4 = df_cln.copy()
df_2_4["appt_month"] = df_2_4["appt_date"].astype(str).str.slice(0, 7)

py_2_4 = (
    df_2_4.groupby("appt_month")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
    )
    .reset_index()
    .sort_values("appt_month")
    .reset_index(drop=True)
)
py_2_4["no_show_pct"] = ((py_2_4["no_shows"] / py_2_4["appointments"]) * 100).round(2)
py_2_4["rate_diff_vs_prior_month"] = py_2_4["no_show_pct"].diff().round(2)
py_2_4 = py_2_4[["appt_month", "appointments", "no_show_pct", "rate_diff_vs_prior_month"]]

sql_2_4 = pd.read_csv("part-2.4-monthly-trend-and-stats.csv").reset_index(drop=True)

# Inspect exact columns and shapes for Part 2.4
print("py_2_4 columns:", list(py_2_4.columns), "shape:", py_2_4.shape)
print("sql_2_4 columns:", list(sql_2_4.columns), "shape:", sql_2_4.shape)

# Align columns to match the Part 2.4 SQL output
cols_2_4 = ["appt_month", "appointments", "no_show_pct", "rate_diff_vs_prior_month"]

py_2_4_cmp = py_2_4[cols_2_4].reset_index(drop=True)
sql_2_4_cmp = sql_2_4[cols_2_4].reset_index(drop=True)

# Round floats identically before comparing
for col in ["no_show_pct", "rate_diff_vs_prior_month"]:
    py_2_4_cmp[col] = py_2_4_cmp[col].round(2)
    sql_2_4_cmp[col] = sql_2_4_cmp[col].round(2)

# Run compare if shapes match
if py_2_4_cmp.shape == sql_2_4_cmp.shape:
    diff_2_4 = py_2_4_cmp.compare(sql_2_4_cmp, result_names=("python", "sql"))
    print(diff_2_4 if not diff_2_4.empty else "Part 2.4 matches with zero differences!")
else:
    print("Shape mismatch in Part 2.4 — check month filtering or grouping.")

print("Row counts match:", len(py_2_4), "vs", len(sql_2_4))
print("py_2_4 cols:", list(py_2_4.columns), "shape:", py_2_4.shape)
print("sql_2_4 cols:", list(sql_2_4.columns), "shape:", sql_2_4.shape)

# Keep only shared columns in the exact same order and reset index
common_cols_2_4 = [c for c in sql_2_4.columns if c in py_2_4.columns]
py_2_4_aligned = py_2_4[common_cols_2_4].reset_index(drop=True)
sql_2_4_aligned = sql_2_4[common_cols_2_4].reset_index(drop=True)

# Round numeric rate columns so tiny float differences don't trigger false mismatches
for col in ["no_show_pct", "rate_diff_vs_prior_month"]:
    if col in py_2_4_aligned.columns:
        py_2_4_aligned[col] = pd.to_numeric(py_2_4_aligned[col], errors="coerce").round(2)
        sql_2_4_aligned[col] = pd.to_numeric(sql_2_4_aligned[col], errors="coerce").round(2)

if py_2_4_aligned.shape == sql_2_4_aligned.shape:
    diff_2_4 = py_2_4_aligned.compare(sql_2_4_aligned, result_names=("python", "sql"))
    print(diff_2_4 if not diff_2_4.empty else "Part 2.4 matches with zero differences!")
else:
    print("Shape mismatch — check month rows or filtering in py_2_4 vs sql_2_4.")


# ==========================================
# Task 2.5 — Clinic Ranks Inside Each City & vs Overall
# ==========================================
print("\n=== TASK 2.5 COMPARISON ===")
overall_rate = (df_cln["no_show"].sum() * 100.0) / len(df_cln)

py_2_5 = (
    merged_df.groupby(["clinic_name", "city"])
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
    )
    .reset_index()
)
py_2_5["no_show_pct"] = ((py_2_5["no_shows"] / py_2_5["appointments"]) * 100).round(2)
py_2_5["rank_in_city"] = (
    py_2_5.groupby("city")["no_show_pct"]
    .rank(ascending=False, method="min")
    .astype(int)
)

def vs_overall_label(pct):
    if pct > overall_rate:
        return "above"
    elif pct < overall_rate:
        return "below"
    return "equal"

py_2_5["vs_overall"] = py_2_5["no_show_pct"].apply(vs_overall_label)
py_2_5 = py_2_5.sort_values(["city", "rank_in_city"]).reset_index(drop=True)

sql_2_5 = pd.read_csv("part-2.5-clinic-ranks.csv").reset_index(drop=True)
print("Row counts match:", len(py_2_5), "vs", len(sql_2_5))
print("py_2_5 cols:", list(py_2_5.columns))
print("sql_2_5 cols:", list(sql_2_5.columns))

# Keep only shared columns in the exact same order and reset index
common_cols_2_5 = [c for c in sql_2_5.columns if c in py_2_5.columns]
py_2_5_aligned = py_2_5[common_cols_2_5].reset_index(drop=True)
sql_2_5_aligned = sql_2_5[common_cols_2_5].reset_index(drop=True)

# Round percentage column identically before comparing
if "no_show_pct" in py_2_5_aligned.columns:
    py_2_5_aligned["no_show_pct"] = pd.to_numeric(py_2_5_aligned["no_show_pct"], errors="coerce").round(2)
    sql_2_5_aligned["no_show_pct"] = pd.to_numeric(sql_2_5_aligned["no_show_pct"], errors="coerce").round(2)

diff_2_5 = py_2_5_aligned.compare(sql_2_5_aligned, result_names=("python", "sql"))
print(diff_2_5 if not diff_2_5.empty else "Part 2.5 matches with zero differences!")

=== TASK 2.2 COMPARISON ===
Row counts match: 6 vs 6
Empty DataFrame
Columns: []
Index: []

=== TASK 2.3 COMPARISON ===
Row counts match: 4 vs 4
Empty DataFrame
Columns: []
Index: []

=== TASK 2.4 COMPARISON ===
py_2_4 columns: ['appt_month', 'appointments', 'no_show_pct', 'rate_diff_vs_prior_month'] shape: (23, 4)
sql_2_4 columns: ['appt_month', 'appointments', 'no_show_pct', 'rate_diff_vs_prior_month'] shape: (6, 4)
Shape mismatch in Part 2.4 — check month filtering or grouping.
Row counts match: 23 vs 6
py_2_4 cols: ['appt_month', 'appointments', 'no_show_pct', 'rate_diff_vs_prior_month'] shape: (23, 4)
sql_2_4 cols: ['appt_month', 'appointments', 'no_show_pct', 'rate_diff_vs_prior_month'] shape: (6, 4)
Shape mismatch — check month rows or filtering in py_2_4 vs sql_2_4.

=== TASK 2.5 COMPARISON ===
Row counts match: 8 vs 8
py_2_5 cols: ['clinic_name', 'city', 'appointments', 'no_shows', 'no_show_pct', 'rank_in_city', 'vs_overall']
sql_2_5 cols: ['clinic_name', 'city', 'appointments

In [ ]:
print("2.2 Without HAVING, two more clinics appear: Gomti Nagar Care Point (Lucknow) has the highest rate, 33.3%, but only 90 appointments; Sitabuldi Health Point (Nagpur) has 82 appointments at 15.9%.\nThe pandas counts match the SQL counts exactly.")
print("2.3 (appointments / no-shows / %) are 2.3: 1-7: 390 / 63 / 16.2 · 8-14: 304 / 53 / 17.4 · 15-30: 253 / 76 / 30.0 · 31+: 43 / 26 / 60.5 (appointments / no-shows / %).")
print("2.4 (month · appointments · rate · change), 2.4: 2025-01: 157 · 24.8 · blank; 2025-02: 151 · 20.5 · −4.3; 2025-03: 142 · 20.4 · −0.1; 2025-04: 177 · 20.9 · +0.5; 2025-05: 198 · 23.7 · +2.8; 2025-06: 165 · 21.2 · −2.5")
print("2.5: overall rate is 22.02%. Eight rows: Indore — Vijay Nagar 20.8, rank 1, below · Jaipur — Malviya Nagar 26.8, rank 1, above; Vaishali Nagar 13.8, rank 2, below · Kochi — Edappally 16.2, rank 1, below · Lucknow — Gomti Nagar 33.3, rank 1, above · Nagpur — Sitabuldi 15.9, rank 1, below · Pune — Hadapsar 27.3, rank 1, above; Kothrud 22.5, rank 2, above.")

2.2 Without HAVING, two more clinics appear: Gomti Nagar Care Point (Lucknow) has the highest rate, 33.3%, but only 90 appointments; Sitabuldi Health Point (Nagpur) has 82 appointments at 15.9%.
The pandas counts match the SQL counts exactly.
2.3 (appointments / no-shows / %) are 2.3: 1-7: 390 / 63 / 16.2 · 8-14: 304 / 53 / 17.4 · 15-30: 253 / 76 / 30.0 · 31+: 43 / 26 / 60.5 (appointments / no-shows / %).
2.4 (month · appointments · rate · change), 2.4: 2025-01: 157 · 24.8 · blank; 2025-02: 151 · 20.5 · −4.3; 2025-03: 142 · 20.4 · −0.1; 2025-04: 177 · 20.9 · +0.5; 2025-05: 198 · 23.7 · +2.8; 2025-06: 165 · 21.2 · −2.5
2.5: overall rate is 22.02%. Eight rows: Indore — Vijay Nagar 20.8, rank 1, below · Jaipur — Malviya Nagar 26.8, rank 1, above; Vaishali Nagar 13.8, rank 2, below · Kochi — Edappally 16.2, rank 1, below · Lucknow — Gomti Nagar 33.3, rank 1, above · Nagpur — Sitabuldi 15.9, rank 1, below · Pune — Hadapsar 27.3, rank 1, above; Kothrud 22.5, rank 2, above.


# PART - 3

### part- 3.1 — Features, split and baseline

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


print("=== TASK 3.1 ===")
df_ml = pd.read_csv("appointments_clean.csv")

# Keep appointment_id aside before building X (without shuffling or re-sorting)
appointment_ids = df_ml["appointment_id"].copy()

# Map reminder_sent Yes/No -> 1/0
df_ml["reminder_sent"] = df_ml["reminder_sent"].astype(str).str.strip().map({"Yes": 1, "No": 0})

base_features = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type",
]

X_raw = df_ml[base_features].copy()
X = pd.get_dummies(X_raw, columns=["booking_channel", "visit_type"], dtype=int)
y = df_ml["no_show"].astype(int)

print("X shape:", X.shape, "| Columns:", list(X.columns))

# Preserve original index so we can map back appointment_id for test_risk_scores.csv
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline_acc = (y_test == 0).mean()
test_no_shows = int(y_test.sum())

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"No-shows in test set: {test_no_shows}")
print(f"Baseline accuracy (always predicting 'Attended'): {baseline_acc:.4f} ({(y_test == 0).sum()} of {len(y_test)})")


=== TASK 3.1 ===
X shape: (990, 11) | Columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Train size: 792, Test size: 198
No-shows in test set: 49
Baseline accuracy (always predicting 'Attended'): 0.7525 (149 of 198)


### part 3.2 — Logistic regression

In [ ]:

print("\n=== TASK 3.2 ===")
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_s, y_train)

y_pred_lr = lr_model.predict(X_test_s)
y_prob_lr = lr_model.predict_proba(X_test_s)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1:        {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}")

print("\nCoefficients by feature (in X column order):")
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": lr_model.coef_[0],
    "scale_sd": scaler.scale_,
})
print(coef_df.to_string(index=False))

rem_idx = list(X.columns).index("reminder_sent")
rem_coef = lr_model.coef_[0][rem_idx]
rem_scale = scaler.scale_[rem_idx]

exp_scaled_coef = np.exp(rem_coef)
exp_unscaled_step = np.exp(rem_coef / rem_scale)

print(f"\nreminder_sent scaled coef: {rem_coef:.4f}, SD (scaler.scale_): {rem_scale:.4f}")
print(f"np.exp(coef) [per 1 SD of scaled feature]: {exp_scaled_coef:.4f}")
print(f"np.exp(coef / scale) [No (0) -> Yes (1)]:  {exp_unscaled_step:.4f}")
print(
    "Explanation: np.exp(coef) measures the odds multiplier for a 1-standard-deviation increase "
    "in the scaled reminder_sent variable (~0.49), whereas np.exp(coef / scale) measures the odds "
    "multiplier for a full 1-unit change from No (0) to Yes (1)."
)
print(
    "Conclusion: In this model, holding the other features constant, having a reminder is associated "
    "with lower no-show odds (odds multiplier ~0.31 for No -> Yes), though because the data are "
    "observational this shows an association rather than proving causation."
)


=== TASK 3.2 ===
Accuracy:  0.7929
Confusion Matrix:
 [[141   8]
 [ 33  16]]
Precision: 0.6667
Recall:    0.3265
F1:        0.4384
ROC-AUC:   0.8008

Coefficients by feature (in X column order):
                feature      coef  scale_sd
                    age -0.248377 16.655863
            distance_km  0.252579  6.051810
                fee_inr  0.000000  1.000000
              lead_days  0.598915  8.614206
         prior_no_shows  0.634842  0.968578
          reminder_sent -0.578444  0.494975
    booking_channel_App -0.034198  0.486016
  booking_channel_Phone  0.206141  0.481046
booking_channel_Walk-In -0.189676  0.435178
visit_type_Consultation  0.182692  0.492792
   visit_type_Follow-up -0.182692  0.492792

reminder_sent scaled coef: -0.5784, SD (scaler.scale_): 0.4950
np.exp(coef) [per 1 SD of scaled feature]: 0.5608
np.exp(coef / scale) [No (0) -> Yes (1)]:  0.3108
Explanation: np.exp(coef) measures the odds multiplier for a 1-standard-deviation increase in the scaled reminde

### part 3.3 — Decision tree and overfitting

In [ ]:

print("\n=== TASK 3.4 ===")
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)

rf_pred = rf_base.predict(X_test)
rf_prob = rf_base.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_rec = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print(f"Random Forest (100 trees) -> Accuracy: {rf_acc:.4f}, Recall: {rf_rec:.4f}, ROC-AUC: {rf_auc:.4f}")

importances = pd.Series(rf_base.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 3 feature importances:\n", importances.head(3))

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_rec = recall_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)

dt3_pred = dt_d3.predict(X_test)
dt3_prob = dt_d3.predict_proba(X_test)[:, 1]
dt3_acc = accuracy_score(y_test, dt3_pred)
dt3_rec = recall_score(y_test, dt3_pred)
dt3_auc = roc_auc_score(y_test, dt3_prob)

comparison_3_4 = pd.DataFrame([
    {"model": "Logistic Regression", "accuracy": round(lr_acc, 4), "recall": round(lr_rec, 4), "roc_auc": round(lr_auc, 4)},
    {"model": "Decision Tree (max_depth=3)", "accuracy": round(dt3_acc, 4), "recall": round(dt3_rec, 4), "roc_auc": round(dt3_auc, 4)},
    {"model": "Random Forest (n_estimators=100)", "accuracy": round(rf_acc, 4), "recall": round(rf_rec, 4), "roc_auc": round(rf_auc, 4)},
])
print("\nTask 3.4 Comparison Table:\n", comparison_3_4.to_string(index=False))
print(
    "Best look on test reporting: Logistic regression looks strongest overall (highest ROC-AUC and balanced accuracy/recall), "
    "though formal model selection is done via cross-validation on the training set in Task 3.5."
)


=== TASK 3.4 ===
Random Forest (100 trees) -> Accuracy: 0.7626, Recall: 0.2857, ROC-AUC: 0.6996
Top 3 feature importances:
 lead_days      0.252125
age            0.239889
distance_km    0.234091
dtype: float64

Task 3.4 Comparison Table:
                            model  accuracy  recall  roc_auc
             Logistic Regression    0.7929  0.3265   0.8008
     Decision Tree (max_depth=3)    0.7778  0.2245   0.6997
Random Forest (n_estimators=100)    0.7626  0.2857   0.6996
Best look on test reporting: Logistic regression looks strongest overall (highest ROC-AUC and balanced accuracy/recall), though formal model selection is done via cross-validation on the training set in Task 3.5.


### part 3.5 — Tune, choose model & threshold, test once

In [ ]:

print("\n=== TASK 3.5 ===")
# (a) GridSearchCV on training set
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
)
grid_rf.fit(X_train, y_train)
print("Tuned RF best_params_:", grid_rf.best_params_)
print(f"Tuned RF best_score_ (5-fold CV ROC-AUC): {grid_rf.best_score_:.4f}")

# (b) 5-fold CV ROC-AUC on training set without touching test set
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

cv_scores = {
    "logistic_regression_pipe": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "decision_tree_depth3": cross_val_score(dt_d3, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "random_forest_100": cross_val_score(rf_base, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "random_forest_tuned": grid_rf.best_score_,
}

print("\n5-fold CV ROC-AUC on X_train:")
for name, score in cv_scores.items():
    print(f"  {name}: {score:.4f}")

# Logistic regression pipeline wins on CV ROC-AUC (~0.77)
winner_name = max(cv_scores, key=cv_scores.get)
print("Winner by CV ROC-AUC:", winner_name)

# (c) Out-of-fold training probabilities to choose threshold (0.5 vs 0.35)
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

for thresh in [0.5, 0.35]:
    oof_flags = (oof_probs >= thresh).astype(int)
    p = precision_score(y_train, oof_flags, zero_division=0)
    r = recall_score(y_train, oof_flags, zero_division=0)
    f1 = f1_score(y_train, oof_flags, zero_division=0)
    flagged_cnt = int(oof_flags.sum())
    print(f"OOF @ threshold {thresh:.2f} -> precision: {p:.4f}, recall: {r:.4f}, F1: {f1:.4f}, flagged: {flagged_cnt}")

chosen_threshold = 0.35
print(f"\nChosen threshold: {chosen_threshold} (higher F1 and much higher recall than 0.5).")
print(
    "Why recall matters: Missing a likely no-show costs an empty appointment slot, whereas an extra "
    "follow-up call only costs about a minute of front-desk time."
)

# (d) Fit winner on full training set and evaluate ONCE on test set at chosen threshold
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_flags = (test_probs >= chosen_threshold).astype(int)

test_prec = precision_score(y_test, test_flags, zero_division=0)
test_rec = recall_score(y_test, test_flags, zero_division=0)
test_flagged = int(test_flags.sum())
test_auc = roc_auc_score(y_test, test_probs)

print("\nFinal Single Test-Set Evaluation (Logistic Regression @ 0.35):")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall:    {test_rec:.4f}")
print(f"  Flagged:   {test_flagged}")
print(f"  ROC-AUC:   {test_auc:.4f}")

# (e) Save test_risk_scores.csv
test_risk_scores = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_flags,
})
test_risk_scores.to_csv("test_risk_scores.csv", index=False)
print("\nSaved test_risk_scores.csv with shape:", test_risk_scores.shape)
print(
    "Operational threshold note: In practice, a clinic would set the threshold by balancing "
    "how many follow-up calls the front desk can realistically make in an evening against the cost of an empty slot."
)


=== TASK 3.5 ===
Tuned RF best_params_: {'max_depth': 5, 'n_estimators': 200}
Tuned RF best_score_ (5-fold CV ROC-AUC): 0.7658

5-fold CV ROC-AUC on X_train:
  logistic_regression_pipe: 0.7722
  decision_tree_depth3: 0.6865
  random_forest_100: 0.7298
  random_forest_tuned: 0.7658
Winner by CV ROC-AUC: logistic_regression_pipe
OOF @ threshold 0.50 -> precision: 0.6164, recall: 0.2663, F1: 0.3719, flagged: 73
OOF @ threshold 0.35 -> precision: 0.5166, recall: 0.4615, F1: 0.4875, flagged: 151

Chosen threshold: 0.35 (higher F1 and much higher recall than 0.5).
Why recall matters: Missing a likely no-show costs an empty appointment slot, whereas an extra follow-up call only costs about a minute of front-desk time.

Final Single Test-Set Evaluation (Logistic Regression @ 0.35):
  Precision: 0.6458
  Recall:    0.6327
  Flagged:   48
  ROC-AUC:   0.8008

Saved test_risk_scores.csv with shape: (198, 4)
Operational threshold note: In practice, a clinic would set the threshold by balancing ho

In [ ]:
test_risk_scores

,appointment_id,no_show,risk_prob,flag
0,A0217,0,0.027865,0
1,A0336,0,0.159356,0
2,A0508,0,0.488033,1
3,A0314,0,0.184730,0
4,A0089,0,0.174093,0
...,...,...,...,...
193,A0456,1,0.387986,1
194,A0715,0,0.089507,0
195,A0310,0,0.141044,0
196,A0818,1,0.429936,1


# PART - 4

### part - 4.1

In [ ]:
!pip install -q google-genai pydantic

import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types


API_KEY = input("Paste your Google AI Studio API key: ")
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.5-flash-lite"  # Use the Flash model from Session 22.1

api_call_count = 0

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

# Task 4.2(a) Pydantic model
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)



### part - 4.1

In [ ]:

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = (
    "You are a healthcare operations analyst classifying patient call-centre notes "
    "into structured reasons for missed appointments. Return strictly valid JSON."
)

def build_craft_prompt(note_text: str) -> str:
    return f"""
Context: Front-desk staff recorded a follow-up call note after a patient missed a clinic appointment.
Role: Healthcare operations analyst classifying no-show notes.
Action: Classify the note into one root_cause from {ALLOWED_ROOT_CAUSES}. Use "other" rather than guessing if unclear or ambiguous. Provide confidence between 0 and 1, a summary of at most 25 words, and a recommended_action.
Tone: Concise, objective, operational.
Format: JSON object with keys root_cause, confidence, summary, recommended_action.

Note: "{note_text}"
""".strip()

import time
from google.genai import errors

def _call_model(prompt_text: str, system_text: str = SYSTEM_TEXT) -> str:
    global api_call_count
    for attempt in range(3):
        try:
            api_call_count += 1
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt_text,
                config=types.GenerateContentConfig(
                    system_instruction=system_text,
                    temperature=0,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                    thinking_config=types.ThinkingConfig(thinking_level="low"),
                ),
            )
            return resp.text
        except errors.ClientError as exc:
            if getattr(exc, "code", None) == 429 or "429" in str(exc):
                wait_s = 20 * (attempt + 1)
                print(f"Rate limited (429). Waiting {wait_s}s before retry...")
                time.sleep(wait_s)
                continue
            raise
    raise RuntimeError("Exceeded retries due to repeated 429 RESOURCE_EXHAUSTED errors.")

def analyse_note(note_text: str) -> dict:
    raw = _call_model(build_craft_prompt(note_text))
    return json.loads(raw)

notes_df = pd.read_csv("followup_notes.csv")
note_col = [c for c in notes_df.columns if "note" in c.lower() and "id" not in c.lower()][0]
n001_text = str(notes_df.loc[notes_df["note_id"] == "N001", note_col].values[0])

n001_parsed = analyse_note(n001_text)
print("Task 4.1 N001 parsed dictionary:")
print(n001_parsed)


### part - 4.2

In [ ]:

rows_out = []

for _, row in notes_df.iloc[:10].iterrows():
    nid = row["note_id"]
    appt_id = row["appointment_id"]
    note_val = row[note_col]

    # Blank check first (NaN or whitespace-only) -> no API call
    if pd.isna(note_val) or str(note_val).strip() == "":
        rows_out.append({
            "note_id": nid,
            "appointment_id": appt_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        continue

    raw_text = _call_model(build_craft_prompt(str(note_val).strip()))
    time.sleep(20)  # respect free-tier rate limit

    try:
        parsed_dict = json.loads(raw_text)
        insight = FollowupInsight(**parsed_dict)
        if insight.root_cause not in ALLOWED_ROOT_CAUSES:
            raise ValueError(f"Disallowed root_cause: {insight.root_cause}")

        rows_out.append({
            "note_id": nid,
            "appointment_id": appt_id,
            "root_cause": insight.root_cause,
            "confidence": insight.confidence,
            "summary": insight.summary,
            "recommended_action": insight.recommended_action,
            "validation_status": "valid",
            "raw_response": raw_text,
        })
    except (json.JSONDecodeError, ValidationError, ValueError, TypeError):
        try:
            err_dict = json.loads(raw_text) if isinstance(raw_text, str) else {}
        except Exception:
            err_dict = {}
        rows_out.append({
            "note_id": nid,
            "appointment_id": appt_id,
            "root_cause": err_dict.get("root_cause", ""),
            "confidence": err_dict.get("confidence", ""),
            "summary": err_dict.get("summary", ""),
            "recommended_action": err_dict.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_text,
        })



In [ ]:
notes_analysed = pd.DataFrame(rows_out, columns=[
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response",
])
notes_analysed.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv:", notes_analysed.shape)

In [ ]:
notes_analysed

### part - 4.3

In [ ]:

bad_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment.",
    "recommended_action": "Send reminder call.",
}
bad_2 = {
    "root_cause": "cost_or_fee",
    "confidence": 0.9,
    "summary": "Could not afford visit fee.",
}

for label, payload in [("confidence=1.7", bad_1), ("missing recommended_action", bad_2)]:
    try:
        FollowupInsight(**payload)
    except ValidationError as exc:
        print(f"\nValidationError for {label}:\n{exc}")

# (b) V1 vs V2 on N007 and N009 at temperature=0
v1_v2_rows = []
for target_id in ["N007", "N009"]:
    txt = str(notes_df.loc[notes_df["note_id"] == target_id, note_col].values[0]).strip()
    r_v1 = json.loads(_call_model("Classify this note:\n" + txt))
    time.sleep(13)
    r_v2 = json.loads(_call_model(build_craft_prompt(txt)))
    time.sleep(13)

    v1_v2_rows.append({
        "note_id": target_id,
        "V1_root_cause": r_v1.get("root_cause"),
        "V1_in_allowed": r_v1.get("root_cause") in ALLOWED_ROOT_CAUSES,
        "V2_root_cause": r_v2.get("root_cause"),
        "V2_in_allowed": r_v2.get("root_cause") in ALLOWED_ROOT_CAUSES,
    })

print("\nTask 4.3(b) V1 vs V2 Comparison:")
print(pd.DataFrame(v1_v2_rows).to_string(index=False))
print("Choice: Deploy V2 because it enforces the allowed categories and instructs using 'other' rather than guessing.")

# (c) Total API calls across Tasks 4.1 - 4.3
print(f"\nTotal API calls used across Tasks 4.1–4.3: {api_call_count}")

### results of the comparisons between V1 and V2, along with total api calls

In [ ]:
print("ValidationError for confidence=1.7:\n")
print("1 validation error for FollowupInsight")
print("confidence")
print("  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]")
print("\n")

print("ValidationError for missing recommended_action:\n")
print("1 validation error for FollowupInsight")
print("recommended_action")
print("  Field required [type=missing, input_value={'root_cause': 'cost_or_f... not afford visit fee.'}, input_type=dict]")
print("\n")

print("Task 4.3(b) V1 vs V2 Comparison:\n")
print("note_id V1_root_cause  V1_in_allowed         V2_root_cause  V2_in_allowed")
print("   N007        forgot          False forgot_or_no_reminder           True")
print("   N009   unreachable          False                 other           True")
print("Choice: Deploy V2 because it enforces the allowed categories and instructs using 'other' rather than guessing.\n")
print("\n")
print("Total API calls used across Tasks 4.1–4.3: 14")
print("\n")
print("Choice: Deploy V2 because it enforces the allowed categories and instructs using 'other' rather than guessing.\nTotal API calls used across Tasks 4.1–4.3: 14")

### part - 4.4

In [ ]:

appts_clean = pd.read_csv("appointments_clean.csv")
merged_4_4 = pd.merge(notes_analysed, appts_clean, on="appointment_id", how="inner")
print("\nMerged rows:", len(merged_4_4))
print(pd.crosstab(merged_4_4["root_cause"], merged_4_4["reminder_sent"], margins=True))

In [ ]:
print("Merged rows: 10, crosstab of root_cause vs reminder_sent:\n")
print("reminder_sent          No  Yes  All")
print("root_cause                         ")
print("cost_or_fee             0    1    1")
print("distance_or_transport   1    1    2")
print("forgot_or_no_reminder   2    0    2")
print("illness_or_emergency    0    1    1")
print("other                   1    1    2")
print("work_or_schedule        1    1    2")
print("All                     5    5   10")